In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
from pathlib import Path
#
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import os

for dirname, _, filenames in os.walk("../input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
%%RecordEvent
import warnings

warnings.filterwarnings("ignore")
import os

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

benchmark_name = "netflix-data-visualization"
df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "netflix-shows" / "netflix_titles.csv"
)
factor = 50
df_cell6 = pd.concat([df] * factor, ignore_index=True)
df.head(3)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

for i in df_cell6.columns:
    null_rate = df_cell6[i].isna().sum() / len(df_cell6) * 100
    if null_rate > 0:
        print("{} null rate: {}%".format(i, round(null_rate, 2)))

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

df_cell6["country"] = df_cell6["country"].fillna(df_cell6["country"].mode()[0])
df_cell6["cast"].replace(np.nan, "No Data", inplace=True)
df_cell6["director"].replace(np.nan, "No Data", inplace=True)
df_cell6.dropna(inplace=True)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###

df_cell6.isnull().sum()

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###

df_cell6.info()

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###

df_cell6["date_added"] = pd.to_datetime(df_cell6["date_added"], errors="coerce")
df_cell6["month_added"] = df_cell6["date_added"].dt.month
df_cell6["new_index"] = df_cell6["date_added"].dt.month_name()
df_cell6["year_added"] = df_cell6["date_added"].dt.year
df_cell6.head(3)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###

x = df_cell6.groupby(["type"])["type"].count()
y = len(df_cell6)
r = (x / y).round(2)
mf_ratio = pd.DataFrame(r).T

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 7 ###

for i_1 in mf_ratio.index:
    _ = int(mf_ratio["Movie"][i_1] * 100)
    _ = mf_ratio["Movie"][i_1] / 2
    _ = mf_ratio["Movie"][i_1] / 2
for i_2 in mf_ratio.index:
    _ = int(mf_ratio["TV Show"][i_2] * 100)
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_8.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 8 ###

df_cell6["count"] = 1
df_cell6["first_country"] = df_cell6["country"].apply(lambda x: x.split(",")[0])
df_cell6["first_country"].head()
ratings_ages = {
    "TV-PG": "Older Kids",
    "TV-MA": "Adults",
    "TV-Y7-FV": "Older Kids",
    "TV-Y7": "Older Kids",
    "TV-14": "Teens",
    "R": "Adults",
    "TV-Y": "Kids",
    "NR": "Adults",
    "PG-13": "Teens",
    "TV-G": "Kids",
    "PG": "Older Kids",
    "G": "Kids",
    "UR": "Adults",
    "NC-17": "Adults",
}
df_cell6["target_ages"] = df_cell6["rating"].replace(ratings_ages)
df_cell6["target_ages"].unique()
df_cell6["genre"] = df_cell6["listed_in"].apply(
    lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
)
df_cell6["first_country"].replace("United States", "USA", inplace=True)
df_cell6["first_country"].replace("United Kingdom", "UK", inplace=True)
df_cell6["first_country"].replace("South Korea", "S. Korea", inplace=True)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 9 ###

data = (
    df_cell6.groupby("first_country")["count"].sum().sort_values(ascending=False)[:10]
)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 10 ###

country_order = df_cell6["first_country"].value_counts()[:11].index
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_q2q3 = (
    df_cell6[["type", "first_country"]]
    .groupby("first_country")["type"]
    .value_counts()
    .unstack()
    .loc[country_order]
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_q2q3 = pd.DataFrame(data_q2q3)
data_q2q3["sum"] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    (data_q2q3.T / data_q2q3["sum"])
    .T[["Movie", "TV Show"]]
    .sort_values(by="Movie", ascending=False)[::-1]
)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 11 ###

order = pd.DataFrame(
    df_cell6.groupby("rating")["count"].sum().sort_values(ascending=False).reset_index()
)
rating_order = list(order["rating"])

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_12.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 12 ###

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
mf = (
    df_cell6.groupby("type")["rating"]
    .value_counts()
    .unstack()
    .sort_index()
    .fillna(0)
    .astype(int)[rating_order]
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    mf = pd.DataFrame(mf)
movie = mf.loc["Movie"]
tv = -mf.loc["TV Show"]

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_13.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 13 ###

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub = (
    df_cell6.groupby("type")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub = pd.DataFrame(data_sub)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 14 ###

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
df_cell6["new_index"] = pd.Categorical(
    df_cell6["new_index"], categories=month_order, ordered=True
)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_15.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 15 ###

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell38 = (
    df_cell6.groupby("type")["new_index"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell38 = pd.DataFrame(data_sub_cell38)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_16.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 16 ###

data_sub_cell38.reset_index()

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_17.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 17 ###

data_sub2 = data_sub_cell38
data_sub2["Value"] = data_sub2["Movie"] + data_sub2["TV Show"]
data_sub2_cell40 = data_sub2.reset_index()
data_sub2_cell40 = data_sub2_cell40.rename(
    columns={data_sub2_cell40.columns[0]: "new_index"}
)
df_polar = data_sub2_cell40.sort_values(by="new_index", ascending=False)
color_map = ["#221f1f" for _ in range(12)]
color_map[0] = color_map[11] = "#b20710"
upperLimit = 30
lowerLimit = 1
labelPadding = 30
max = df_polar["Value"].max()
slope = (max - lowerLimit) / max
heights = slope * df_polar.Value + lowerLimit
width = 2 * np.pi / len(df_polar.index)
indexes = list(range(1, len(df_polar.index) + 1))
angles = [element * width for element in indexes]
angles

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_18.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 18 ###

import matplotlib.colors

cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "", ["#221f1f", "#b20710", "#f5f5f1"]
)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_19.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 19 ###


def genre_heatmap(df, title):
    df["genre"] = df["listed_in"].apply(
        lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
    )
    Types = []
    for i_3 in df["genre"]:
        Types += i_3
    Types = set(Types)
    print("There are {} types in the Netflix {} Dataset".format(len(Types), title))
    test_1 = df["genre"]


df_tv = df_cell6[df_cell6["type"] == "TV Show"]
df_movies = df_cell6[df_cell6["type"] == "Movie"]
genre_heatmap(df_movies, "Movie")

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 20 ###

data_cell45 = (
    df_cell6.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data_cell45 = data_cell45["first_country"]
df_heatmap = df_cell6.loc[df_cell6["first_country"].isin(data_cell45)]

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_21.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 21 ###

df_heatmap_cell46 = pd.crosstab(
    df_heatmap["first_country"], df_heatmap["target_ages"], normalize="index"
).T

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 22 ###

df_movies
df_tv
data_cell51 = (
    df_movies.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data_cell51 = data_cell51["first_country"]
df_loli = df_movies.loc[df_movies["first_country"].isin(data_cell51)]
loli = df_loli.groupby("first_country")[["release_year", "year_added"]].mean().round()
ordered_df = loli.sort_values(by="release_year")
ordered_df_rev = loli.sort_values(by="release_year", ascending=False)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_23.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 23 ###

us_ind = df_cell6[
    (df_cell6["first_country"] == "USA") | (df_cell6["first_country"] == "India")
]
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell54 = (
    df_cell6.groupby("first_country")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["USA", "India"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell54 = pd.DataFrame(data_sub_cell54)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_24.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 24 ###

us_ind_cell57 = df_cell6[
    (df_cell6["first_country"] == "USA") | (df_cell6["first_country"] == "India")
]
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = df_cell6._to_pandas()
data_sub_cell57 = (
    df_cell6.groupby("first_country")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["USA", "India"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df_cell6 = pd.DataFrame(df_cell6)
    data_sub_cell57 = pd.DataFrame(data_sub_cell57)
data_sub_cell57.insert(0, "base", np.zeros(len(data_sub_cell57)))
data_sub_cell57 = data_sub.add(-us_ind_cell57["year_added"].value_counts() / 2, axis=0)

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_25.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 25 ###

import matplotlib

text = (
    str(list(df_cell6["title"]))
    .replace(",", "")
    .replace("[", "")
    .replace("'", "")
    .replace("]", "")
    .replace(".", "")
)